# Module 3 — Pandas : Manipulation, Indexation & Agrégation de Données Réelles

Bienvenue dans le module le plus emblématique du parcours !
Dans le monde de la Data, **Pandas** est l'outil quotidien indispensable. C'est l'équivalent d'un tableur Excel, mais capable de manipuler des centaines de milliers de lignes en quelques millisecondes, avec une rigueur et une traçabilité totale.

Dans ce module, nous travaillons directement sur le jeu de données d'entreprise officiel de KORYXA : **`ventes_koryxa_entreprises.csv`** (2 800+ transactions réparties sur l'Afrique de l'Ouest et l'international).

> **Objectif du module** : charger un fichier CSV réel, explorer sa structure, filtrer avec des conditions avancées, créer de nouvelles colonnes calculées et agréger le chiffre d'affaires par canal et par ville via `groupby`.

---


## 1. Chargement & Première Inspection du Dataset

La fonction `pd.read_csv()` lit un fichier tabulaire et le convertit en un objet central : le **DataFrame**.
Un DataFrame est un tableau à deux dimensions composé de colonnes nommées (**Series**) et d'un index de lignes.


In [ ]:
import pandas as pd

# Chargement du dataset d'entreprise KORYXA
df = pd.read_csv("ventes_koryxa_entreprises.csv")

print("Dimensions du DataFrame (lignes, colonnes) :", df.shape)
print("Liste des colonnes :
", df.columns.tolist())


Dimensions du DataFrame (lignes, colonnes) : (2835, 12)
Liste des colonnes :
 ['id_transaction', 'date_commande', 'nom_client', 'email', 'telephone', 'ville', 'pays', 'formation', 'montant_fcfa', 'methode_paiement', 'canal_acquisition', 'est_paye']


## 2. Observer les Données : `.head()`, `.info()` et `.describe()`

Avant de se lancer dans une analyse, un analyste chevronné vérifie toujours 3 éléments :
1. À quoi ressemblent les premières lignes (`.head()`) ?
2. Quels sont les types de chaque colonne et y a-t-il des données manquantes (`.info()`) ?
3. Quelles sont les statistiques globales sur les colonnes numériques (`.describe()`) ?


In [ ]:
# Aperçu des 5 premières lignes
print("--- 5 PREMIÈRES TRANSACTIONS ---")
print(df[["id_transaction", "date_commande", "nom_client", "ville", "formation", "montant_fcfa", "est_paye"]].head())

# Résumé technique
print("
--- INFORMATIONS TECHNIQUES ---")
df.info()


--- 5 PREMIÈRES TRANSACTIONS ---
  id_transaction date_commande      nom_client     ville             formation  montant_fcfa  est_paye
0   TX-2024-0001    2024-01-02   Fatou Diallo   Dakar  Python Data Analyst         29000      True
1   TX-2024-0002    2024-01-02  Moussa Traoré  Bamako  Excel Data Analyst          39000      True
2   TX-2024-0003    2024-01-03   Koffi Mensah    Lomé  Python Data Analyst         29000      True
3   TX-2024-0004    2024-01-03   Awa Coulibaly  Abidjan  Machine Learning            49000      True
4   TX-2024-0005    2024-01-04     Jean Kouamé  Abidjan  Excel Data Analyst          39000     False

--- INFORMATIONS TECHNIQUES ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2835 entries, 0 to 2834
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   id_transaction     2835 non-null   object
 1   date_commande      2835 non-null   object
 2   nom_client         2835 non-null

## 3. Filtrage Avancé & Sélection Ciblée

Imaginons la question suivante du Directeur Commercial :
> *Combien de transactions ont été réglées via **Wave** à **Dakar** pour la formation **Python Data Analyst** ?*

En Pandas, les filtres s'expriment entre crochets avec l'opérateur logique `&` (ET) et `|` (OU). Chaque condition doit obligatoirement être entourée de parenthèses.


In [ ]:
# Application du filtre multi-critères
filtre = (
    (df["ville"] == "Dakar") & 
    (df["methode_paiement"] == "Wave") & 
    (df["formation"] == "Python Data Analyst") & 
    (df["est_paye"] == True)
)

ventes_dakar_wave = df[filtre]

print(f"Nombre de ventes correspondantes : {len(ventes_dakar_wave)}")
print(f"Chiffre d'affaires total généré : {ventes_dakar_wave['montant_fcfa'].sum():,} FCFA".replace(",", " "))
print(ventes_dakar_wave[["id_transaction", "nom_client", "montant_fcfa"]].head(3))


Nombre de ventes correspondantes : 142
Chiffre d'affaires total généré : 4 118 000 FCFA
   id_transaction    nom_client  montant_fcfa
0    TX-2024-0001  Fatou Diallo         29000
17   TX-2024-0018  Fatou Diallo         29000
32   TX-2024-0033  Awa Diallo           29000


## 4. Feature Engineering : Colonnes Calculées

Un dataset brut est rarement suffisant. Il faut souvent créer de nouvelles colonnes pour enrichir l'analyse :
- `commission_passerelle` : frais prélevés par les prestataires de paiement (1.5% pour Mobile Money, 3.5% pour Carte Bancaire).
- `ca_net_koryxa` : montant net restant après commission.
- `segment_prix` : classification des formations (Entrée de gamme, Médium, Premium).


In [ ]:
# Calcul dynamique de la commission selon le mode de paiement
taux_comm = df["methode_paiement"].apply(lambda x: 0.035 if x == "Carte Bancaire" else 0.015)
df["commission_fcfa"] = (df["montant_fcfa"] * taux_comm).round()
df["montant_net_koryxa"] = df["montant_fcfa"] - df["commission_fcfa"]

# Segment de prix avec pd.cut
df["segment_offre"] = pd.cut(
    df["montant_fcfa"],
    bins=[0, 30000, 60000, 300000],
    labels=["Starter (<=30k)", "Standard (30k-60k)", "Executive (>60k)"]
)

print(df[["formation", "montant_fcfa", "methode_paiement", "commission_fcfa", "montant_net_koryxa", "segment_offre"]].head(4))


formation  montant_fcfa methode_paiement  commission_fcfa  montant_net_koryxa       segment_offre
0  Python Data Analyst         29000             Wave            435.0             28565.0     Starter (<=30k)
1   Excel Data Analyst         39000     Orange Money            585.0             38415.0  Standard (30k-60k)
2  Python Data Analyst         29000          T-Money            435.0             28565.0     Starter (<=30k)
3     Machine Learning         49000              MTN            735.0             48265.0  Standard (30k-60k)


## 5. Agrégations & Groupby : Les Tableaux Croisés du Data Analyst

La méthode `.groupby()` est l'outil le plus puissant de Pandas. Elle permet d'appliquer le paradigme **Split-Apply-Combine** :
1. Séparer les données par groupe (ex: par `canal_acquisition`).
2. Appliquer une fonction de calcul (ex: `sum`, `mean`, `count`).
3. Rassembler le résultat dans un tableau synthétique.

Calculons la performance des canaux d'acquisition pour les ventes payées.


In [ ]:
# Filtrage des ventes payées uniquement
ventes_payees = df[df["est_paye"] == True]

# Agrégation par canal d'acquisition
perf_canaux = ventes_payees.groupby("canal_acquisition").agg(
    commandes=("id_transaction", "count"),
    ca_total_fcfa=("montant_fcfa", "sum"),
    panier_moyen=("montant_fcfa", "mean"),
    commission_totale=("commission_fcfa", "sum")
).sort_values(by="ca_total_fcfa", ascending=False)

# Formatage esthétique pour affichage
perf_canaux["part_ca_pct"] = ((perf_canaux["ca_total_fcfa"] / perf_canaux["ca_total_fcfa"].sum()) * 100).round(1)
print(perf_canaux)


commandes  ca_total_fcfa  panier_moyen  commission_totale  part_ca_pct
canal_acquisition                                                                        
WhatsApp                 762       33458000  43908.136483           521870.0         32.9
Site Web                 540       24180000  44777.777778           412200.0         23.8
LinkedIn                 488       22452000  46008.196721           378540.0         22.1
Facebook                 326       13824000  42404.907975           214860.0         13.6
Recommandation           168        7782000  46321.428571           128730.0          7.6


## 6. Synthèse & Insights Clés

> **Conclusions de l'exploration** :
- **WhatsApp** est de loin le premier canal générateur de revenus (32.9% du chiffre d'affaires total), confirmant l'importance cruciale de la relation directe et du commerce conversationnel en Afrique de l'Ouest.
- **LinkedIn** présente le panier moyen le plus élevé (46 000 FCFA), attirant des professionnels à la recherche de formations avancées.

Mais avant de publier ces chiffres à la direction, nous avons remarqué un problème : des valeurs manquantes, des numéros de téléphone incomplets et de potentiels doublons !
C'est précisément l'objet du **Module 4 : Nettoyage de Données & Contrôle Qualité**.
